In [1]:
# notebook : diagnostic_semdetect_ids.ipynb — cellule 1 : chargement brut
import pandas as pd
from datasets import load_dataset, concatenate_datasets

from src.data.dataset_semdetect import REPO_ID, CONFERENCES, YEARS

hf_splits = [f"{prefix}_{y}" for prefix in CONFERENCES.values() for y in YEARS]
raw = concatenate_datasets(
    [load_dataset(REPO_ID, "reviews", split=s) for s in hf_splits]
).to_pandas()

df = raw.copy()
df["year"] = df["year"].astype(str).str[:4]
for col in ["paper_id", "submission_id", "conference", "review_id", "og_review_id",
            "author", "class", "rating", "clean_review", "split"]:
    df[col] = df[col].fillna("").astype(str)

df["review-id"] = df["conference"] + "-" + df["year"] + "-" + df["submission_id"] + "-" + df["review_id"]
print(len(df), "lignes")

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-28 15:04:34.171 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Sem-Detect/ML_Conferences-Peer-Reviews/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Sem-Detect/ML_Conferences-Peer-Reviews/18b74b551670bff6b04e57570232a1694997863b/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Sem-Detect/ML_Conferences-Peer-Reviews/resolve/18b74b551670bff6b04e57570232a1694997863b/ML_Conferences-Peer-Reviews.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.

22165 lignes


In [2]:
# notebook : diagnostic_semdetect_ids.ipynb — cellule 2 : isolement des doublons
dups = df[df["review-id"].duplicated(keep=False)].sort_values("review-id")
print(f"{len(dups)} lignes impliquées, {dups['review-id'].nunique()} clés distinctes")
print(dups.groupby(["split", "class"]).size())
dups[["review-id", "split", "class", "author", "og_review_id", "rating"]]

110 lignes impliquées, 55 clés distinctes
split  class  
test   ai         16
       human       8
       rewrite    64
train  human      22
dtype: int64


,review-id,split,class,author,og_review_id,rating
11196,NeurIPS-2021-k4gHgRyPbYS-0,test,human,human,,Borderline
14236,NeurIPS-2021-k4gHgRyPbYS-0,test,human,human,,Borderline
14237,NeurIPS-2021-k4gHgRyPbYS-1,test,human,human,,Borderline
11197,NeurIPS-2021-k4gHgRyPbYS-1,test,human,human,,Borderline
11198,NeurIPS-2021-k4gHgRyPbYS-2,test,human,human,,Borderline
...,...,...,...,...,...,...
13837,NeurIPS-2021-vrXuRmaU_jM-0,train,human,human,,Weak Reject
14102,NeurIPS-2021-vrXuRmaU_jM-1,train,human,human,,Borderline
13838,NeurIPS-2021-vrXuRmaU_jM-1,train,human,human,,Borderline
14103,NeurIPS-2021-vrXuRmaU_jM-2,train,human,human,,Weak Accept


In [3]:
# notebook : diagnostic_semdetect_ids.ipynb — cellule 3 : caractérisation H1 / H2 / H3
def _uniq(s: pd.Series) -> tuple:
    return tuple(sorted(set(s)))

agg = dups.groupby("review-id").agg(
    n=("class", "size"),
    classes=("class", _uniq),
    authors=("author", _uniq),
    n_content=("clean_review", "nunique"),
    n_split=("split", "nunique"),
    n_og=("og_review_id", "nunique"),
    n_rating=("rating", "nunique"),
)
agg

,n,classes,authors,n_content,n_split,n_og,n_rating
review-id,,,,,,,
NeurIPS-2021-k4gHgRyPbYS-0,2,"(human,)","(human,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-1,2,"(human,)","(human,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-2,2,"(human,)","(human,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-3,2,"(human,)","(human,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-ai_deepseek.v3-v1:0_Borderline_45bc2276,2,"(ai,)","(deepseek.v3-v1:0,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-ai_deepseek.v3-v1:0_Borderline_93bb1d71,2,"(ai,)","(deepseek.v3-v1:0,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-ai_gemini-2.5-flash_Borderline_2f58ad84,2,"(ai,)","(gemini-2.5-flash,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-ai_gemini-2.5-flash_Borderline_fa619c84,2,"(ai,)","(gemini-2.5-flash,)",1,1,1,1
NeurIPS-2021-k4gHgRyPbYS-ai_gemini-2.5-pro_Borderline_b9a3d2e5,2,"(ai,)","(gemini-2.5-pro,)",1,1,1,1


In [4]:
# notebook : diagnostic_semdetect_ids.ipynb — cellule 4 : format des review_id et clés alternatives
print(df.groupby("class")["review_id"].apply(lambda s: s.sample(3, random_state=0).tolist()))

print("\nDoublons stricts (toutes colonnes) :", int(raw.astype(str).duplicated().sum()))

candidate_keys = [
    ["submission_id", "review_id"],
    ["submission_id", "class", "review_id"],
    ["submission_id", "author", "review_id"],
    ["submission_id", "author", "og_review_id", "review_id"],
    ["submission_id", "review_id", "clean_review"],
]
for key in candidate_keys:
    print(f"{key!s:<60} -> {int(df.duplicated(subset=key).sum())} doublons")

class
ai         [ai_qwen.qwen3-235b-a22b-2507-v1:0_Borderline_...
human                                              [2, 1, 0]
rewrite    [rewrite_1_gemini-2.5-pro_1634a386, rewrite_3_...
Name: review_id, dtype: object

Doublons stricts (toutes colonnes) : 44
['submission_id', 'review_id']                               -> 55 doublons
['submission_id', 'class', 'review_id']                      -> 55 doublons
['submission_id', 'author', 'review_id']                     -> 55 doublons
['submission_id', 'author', 'og_review_id', 'review_id']     -> 55 doublons
['submission_id', 'review_id', 'clean_review']               -> 44 doublons


In [5]:
# notebook : diagnostic_semdetect_ids.ipynb — cellule 5 : nature des 11 collisions
import difflib

cols_no_key = [c for c in df.columns if c != "review-id"]
dedup = df.drop_duplicates(subset=cols_no_key)
coll = dedup[dedup["review-id"].duplicated(keep=False)].sort_values("review-id")

print(coll.groupby(["split", "class", "author"]).size(), "\n")
for rid, g in coll.groupby("review-id"):
    a, b = g["clean_review"].tolist()[:2]
    ratio = difflib.SequenceMatcher(None, a, b).ratio()
    print(f"{rid:<45} ratings={g['rating'].tolist()} len=({len(a)}, {len(b)}) similarité={ratio:.3f}")

split  class  author
train  human  human     22
dtype: int64 

NeurIPS-2021-sYNr-OqGC9m-0                    ratings=['Borderline', 'Borderline'] len=(951, 4996) similarité=0.021
NeurIPS-2021-sYNr-OqGC9m-1                    ratings=['Weak Accept', 'Borderline'] len=(5789, 2273) similarité=0.024
NeurIPS-2021-sYNr-OqGC9m-2                    ratings=['Borderline', 'Weak Accept'] len=(2822, 713) similarité=0.011
NeurIPS-2021-sYNr-OqGC9m-3                    ratings=['Weak Accept', 'Weak Reject'] len=(4102, 2193) similarité=0.017
NeurIPS-2021-ueGDv64HmO-0                     ratings=['Weak Accept', 'Borderline'] len=(2576, 2581) similarité=0.023
NeurIPS-2021-ueGDv64HmO-1                     ratings=['Weak Reject', 'Borderline'] len=(1773, 2713) similarité=0.044
NeurIPS-2021-ueGDv64HmO-2                     ratings=['Borderline', 'Weak Reject'] len=(4659, 1827) similarité=0.026
NeurIPS-2021-ueGDv64HmO-3                     ratings=['Borderline', 'Weak Accept'] len=(2113, 6240) similarité=0

In [6]:
# notebook : diagnostic_contamination.ipynb — cellule 0 : silence des logs HTTP
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

In [7]:
# notebook : diagnostic_contamination.ipynb — cellule 1 : recouvrement Sem-Detect test / APRDB train
import re

import numpy as np
import pandas as pd

from src.data.dataset_aprdb import AprdbDataset
from src.data.dataset_semdetect import SemDetectDataset

# Copie exacte de `dataset_train` de train_sage_gru_2_cat (même seed -> même sous-échantillon)
APRDB_TRAIN_CFG = {
    "name": "aprdb",
    "description": "AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt",
    "conference": None,
    "date": None,
    "keep_meta": True,
    "subset_size": 10000,
    "split": "calibration",
    "with_pdf": False,
    "seed": 0,
}
SEMDETECT_TEST_CFG = {
    "name": "sem_detect",
    "split": "test",
    "conference": None,
    "date": None,
    "generator": None,
    "label_mode": "human_vs_ai",
    "balance": "none",
    "subset_size": None,
    "no_duplicate": True,
    "keep_meta": True,
    "with_paper": False,
    "keep_claims": False,
    "seed": 0,
}

_WORD = re.compile(r"[a-z0-9]+")


def shingles(text: str, n: int = 12) -> set:
    toks = _WORD.findall(str(text).lower())
    return {" ".join(toks[i:i + n]) for i in range(max(len(toks) - n + 1, 0))}


aprdb = AprdbDataset(config=APRDB_TRAIN_CFG).to_pandas()
aprdb_human = aprdb[aprdb["label"] == 0]
ref = set().union(*(shingles(t) for t in aprdb_human["content"]))
print(f"APRDB train : {len(aprdb_human)} reviews humaines, {len(ref)} shingles")

sd = SemDetectDataset(config=SEMDETECT_TEST_CFG).to_pandas()
sd_human = sd[sd["label_3c"] == 0].copy()


def overlap(text: str) -> float:
    s = shingles(text)
    return len(s & ref) / len(s) if s else np.nan


sd_human["overlap"] = sd_human["content"].map(overlap)
sd_human["contaminated"] = sd_human["overlap"] >= 0.5

print(sd_human["overlap"].describe(), "\n")
print(sd_human.groupby(["conference", "year"])["contaminated"].agg(["sum", "count", "mean"]))

/data2/cyaiche/cache_dir
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 10000, 'split': 'calibration', 'with_pdf': False, 'seed': 0}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_4fbe88291ebaf69cc7d4bd78214b8d51.pt


INFO:src.data.dataset_semdetect:Loading Sem-Detect dataset from Sem-Detect/ML_Conferences-Peer-Reviews (revision=None)


APRDB train : 3017 reviews humaines, 1233896 shingles


split  class    reason          
train  human    collision           22
       rewrite  rewrite_orphelin    88
dtype: int64
INFO:src.data.dataset_semdetect:Déduplication : 0 doublons supprimés
INFO:src.data.dataset_semdetect:Sem-Detect | split=['test'] conf=['ICLR', 'NeurIPS'] years=['2021', '2022'] label_mode=human_vs_ai balance=none | 1970/22165 reviews
INFO:src.data.dataset_semdetect:Répartition class x label :
label       0     1
label_3c           
ai          0  1364
human     606     0


count    606.000000
mean       0.075385
std        0.243706
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: overlap, dtype: float64 

                 sum  count      mean
conference year                      
ICLR       2021   22    148  0.148649
           2022    0    161  0.000000
NeurIPS    2021   31    112  0.276786
           2022    0    185  0.000000


In [8]:
# notebook : diagnostic_contamination.ipynb — cellule 2 : export des review-id contaminés
import os
from pathlib import Path

out_dir = Path(os.environ["CACHE_DIR"]) / "contamination"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "semdetect_test_vs_aprdb_calibration_2016-2021.txt"

contaminated_ids = sorted(sd_human.loc[sd_human["contaminated"], "review-id"])
out_path.write_text("\n".join(contaminated_ids) + "\n", encoding="utf-8")
print(f"{len(contaminated_ids)} ids -> {out_path}")

53 ids -> /data2/cyaiche/cache_dir/contamination/semdetect_test_vs_aprdb_calibration_2016-2021.txt


In [9]:
# notebook : diagnostic_artifacts.ipynb — cellule 1 : inventaire des artefacts SAGE
import os
from pathlib import Path

import joblib
import pandas as pd
import yaml

results = Path(os.environ["CACHE_DIR"]) / "results"
rows = []
for art_path in sorted(results.glob("*/artifact.joblib")):
    art = joblib.load(art_path)
    cfg = yaml.safe_load(art.get("cfg", "")) or {}
    model_cfg = cfg.get("training", {}).get("model", {})
    spec = art.get("feature_spec") or {}
    rows.append({
        "hash": art_path.parent.name,
        "model_name": model_cfg.get("name"),
        "feature_spec_cfg": model_cfg.get("feature_spec"),
        "feature_spec_name": spec.get("name") if isinstance(spec, dict) else None,
        "n_features": art.get("model_kwargs", {}).get("n_features"),
        "train_split": cfg.get("dataset_train", {}).get("split"),
        "auc_val": art.get("auc"),
        "ctx_type": type(art.get("ctx")).__name__,
        "mtime": pd.Timestamp(art_path.stat().st_mtime, unit="s"),
    })

pd.DataFrame(rows).sort_values("mtime", ascending=False)

,hash,model_name,feature_spec_cfg,feature_spec_name,n_features,train_split,auc_val,ctx_type,mtime
27,7dcd88721a208c75f42bf5952cddb949,sage_gru_1,v3_merge_suggestion,v3_merge_suggestion,6.0,calibration,0.942834,str,2026-09-28 08:12:47.606964111
10,2285fd56da063759d6e6adda84e1b588,sage_gru_1,v3_merge_suggestion,v3_merge_suggestion,6.0,calibration,0.924510,str,2026-09-28 08:12:12.540963173
32,8b2c5cbcb41fde417419155bd3ff3b7c,sage_gru_1,v3_merge_suggestion,v3_merge_suggestion,6.0,calibration,0.914193,str,2026-09-28 08:11:37.854962349
17,518c38580ff73647450233fe098e2349,sage_gru_1,v3_merge_suggestion,v3_merge_suggestion,6.0,calibration,0.930582,str,2026-09-28 08:11:03.264961481
40,af30be005359d88a2be07254b2909aa7,sage_gru_1,v3_merge_suggestion,v3_merge_suggestion,6.0,calibration,0.934924,str,2026-09-28 08:10:27.897960663
3,085cdac5ae585cc933c391e8c0cd52c6,sage_gru_1,v3_merge_question,v3_merge_question,6.0,calibration,0.936820,str,2026-09-28 08:09:53.341959715
48,eb572109f9d1c60640b6e413a2935d87,sage_gru_1,v3_merge_question,v3_merge_question,6.0,calibration,0.932491,str,2026-09-28 08:09:19.099958897
11,22898edca8acdb3f072582b50b5a4779,sage_gru_1,v3_merge_question,v3_merge_question,6.0,calibration,0.925814,str,2026-09-28 08:08:44.301958084
47,e1eb559063827476bfaeefa4619ee48a,sage_gru_1,v3_merge_question,v3_merge_question,6.0,calibration,0.934077,str,2026-09-28 08:08:09.750957251
41,b19c8b3ec771903bf474868b5f06d52e,sage_gru_1,v3_merge_question,v3_merge_question,6.0,calibration,0.931221,str,2026-09-28 08:07:34.838956356


In [1]:
# notebook : validation_semdetect_paired.ipynb — cellule 1 : invariants split + appariement
import logging

import pandas as pd

from src.data.dataset_semdetect import SemDetectDataset

logging.getLogger("httpx").setLevel(logging.WARNING)

BASE = {
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews",
    "revision": "18b74b551670bff6b04e57570232a1694997863b",
    "conference": None, "date": None, "generator": None,
    "subset_size": None, "val_fraction": 0.1, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": False, "keep_claims": False,
    "exclude_ids_path": None, "seed": 0,
}


def load(**kw) -> pd.DataFrame:
    return SemDetectDataset(config={**BASE, **kw}).to_pandas()


tr = load(split="train", label_mode="three_class", balance="paired")
va = load(split="val", label_mode="three_class", balance="paired")
te = load(split="test", label_mode="human_vs_ai", balance="none", val_fraction=None)

# 1. splits disjoints au niveau papier, test intact
p_tr, p_va, p_te = set(tr.submission_id), set(va.submission_id), set(te.submission_id)
assert not (p_tr & p_va) and not (p_tr & p_te) and not (p_va & p_te), "fuite de papiers entre splits"
assert len(te) == 1970, f"test modifié : {len(te)}"
print(f"papiers : train={len(p_tr)} val={len(p_va)} test={len(p_te)}")

# 2. équilibre exact par classe réelle
for name, d in [("train", tr), ("val", va)]:
    print(name, d["class"].value_counts().to_dict())
    assert d["class"].value_counts().nunique() == 1, f"{name} déséquilibré"

# 3. contraintes d'appariement
for name, d in [("train", tr), ("val", va)]:
    g = d.set_index(["pair-id", "class"])
    h = g.xs("human", level="class")
    a = g.xs("ai", level="class")
    r = g.xs("rewrite", level="class")
    assert (a.loc[h.index, "submission_id"] == h["submission_id"]).all(), "IA d'un autre papier"
    assert (a.loc[h.index, "rating"] == h["rating"]).all(), "IA d'une autre note"
    assert (r.loc[h.index, "source-review-id"] == h["review-id"]).all(), "réécriture d'une autre humaine"
    assert a["review-id"].is_unique, "review IA réutilisée"
    print(f"{name} : appariement OK ({len(h)} groupes)")

# 4. répartition des notes par classe (doit être identique par construction)
print(pd.crosstab(tr["class"], tr["rating"]))

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-29 12:00:42.933 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub
INFO:src.data.dataset_semdetect:Loading Sem-Detect dataset from Sem-Detect/ML_Conferences-Peer-Reviews (revision=18b74b551670bff6b04e57570232a1694997863b)
split  class    reason          
train  human    collision           22
       rewrite  rewrite_orphelin    88
dtype: int64
INFO:src.data.dataset_semdetect:Split validation (val_fraction=0.1, val_seed=0) : papiers par split = {'test': 159, 'train': 571, 'val': 66}
INFO:src.data.dataset_semdetect:Déduplication : 61 doublons supprimés
INFO:src.data.dataset_semdetect:Appariement : 2176 groupes | humaines initiales 2178, sans réécriture 0

papiers : train=568 val=66 test=159
train {'rewrite': 2176, 'human': 2176, 'ai': 2176}
val {'rewrite': 249, 'ai': 249, 'human': 249}
train : appariement OK (2176 groupes)
val : appariement OK (249 groupes)
rating   Accept  Borderline  Reject  Weak Accept  Weak Reject
class                                                        
ai          271        1131     148          467          159
human       271        1131     148          467          159
rewrite     271        1131     148          467          159
